# 🎙️ Nabih AI - XTTS-v2 Voice Cloning Cloud Server
### High-Speed Arabic Voice Synthesis with Free GPU (Nvidia T4)
---
This notebook runs **Coqui XTTS-v2** on Google Colab's free GPU and exposes a public **Cloudflare Tunnel API**.
Project **Nabih** on your desktop connects to this server to speak with realistic cloned voices at $0 cost!

### 1️⃣ Check GPU Acceleration
Make sure your Colab runtime is set to **GPU** (`Runtime -> Change runtime type -> T4 GPU`).

In [ ]:
!nvidia-smi

### 2️⃣ Install Dependencies & Cloudflared Tunnel
Installs `coqui-tts`, `fastapi`, `uvicorn`, and the Cloudflare Tunnel client.

In [ ]:
!pip install -q coqui-tts fastapi uvicorn nest-asyncio soundfile edge-tts
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print('✅ Environment & Cloudflared installed successfully!')

### 3️⃣ Setup Voice Sample for Cloning
You can upload your own 6-second voice sample as `my_voice.wav` in the Colab file browser on the left.
If not uploaded, the script generates a clean Egyptian reference sample automatically.

In [ ]:
import os

SPEAKER_FILE = 'my_voice.wav'
if not os.path.exists(SPEAKER_FILE):
    print('ℹ️ "my_voice.wav" not found. Generating default clean Egyptian voice sample...')
    !edge-tts --text "أهلاً يا برنس، أنا نبيه مساعدك الذكي ومستعد لكل طلباتك." --voice "ar-EG-ShakirNeural" --write-media my_voice.wav
    print('✅ Default voice reference created: my_voice.wav')
else:
    print('🎯 Using your uploaded voice sample: my_voice.wav')

### 4️⃣ Load XTTS-v2 Model on GPU
Downloads the official multilingual XTTS-v2 model and places it in GPU VRAM.

In [ ]:
import torch
from TTS.api import TTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🚀 Loading XTTS-v2 on {device.upper()}...')

tts_engine = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
print('✅ XTTS-v2 loaded in GPU memory and ready!')

### 5️⃣ Launch FastAPI Server with Cloudflare Tunnel
Runs the FastAPI server on port 8000 and prints your public **`trycloudflare.com`** URL to copy into Nabih's `.env`!

In [ ]:
import io
import re
import time
import subprocess
import soundfile as sf
from fastapi import FastAPI, Response
from pydantic import BaseModel
import uvicorn
import threading

app = FastAPI(title='Nabih XTTS-v2 Voice Server')

class SynthesizeRequest(BaseModel):
    text: str
    language: str = 'ar'
    speaker: str = 'my_voice.wav'

@app.get('/health')
def health():
    return {'status': 'healthy', 'device': device, 'model': 'xtts_v2'}

@app.post('/synthesize')
def synthesize(req: SynthesizeRequest):
    speaker_path = req.speaker if os.path.exists(req.speaker) else 'my_voice.wav'
    t0 = time.perf_counter()
    # XTTS inference on GPU
    wav = tts_engine.tts(text=req.text, speaker_wav=speaker_path, language=req.language)
    latency = (time.perf_counter() - t0) * 1000

    bio = io.BytesIO()
    sf.write(bio, wav, 24000, format='WAV')
    bio.seek(0)

    print(f'⚡ Synthesized "{req.text[:25]}..." in {latency:.0f}ms')
    return Response(content=bio.read(), media_type='audio/wav')

# Start FastAPI in a background thread
def run_uvicorn():
    uvicorn.run(app, host='127.0.0.1', port=8000, log_level='warning')

server_thread = threading.Thread(target=run_uvicorn, daemon=True)
server_thread.start()
time.sleep(2)

# Start Cloudflared Tunnel
print('🌐 Starting Cloudflare Public Tunnel...')
tunnel_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_url = None
for line in tunnel_process.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

print('\n' + '='*65)
print(f'🎉 YOUR NABIH XTTS ENDPOINT IS READY:')
print(f'👉 {tunnel_url}')
print('='*65)
print('Copy this URL and paste it in Nabih project .env:')
print(f'XTTS_API_URL={tunnel_url}')
print(f'TTS_PROVIDER=xtts')
print('='*65 + '\n')